### Ingestión del archivo "movie_genre.json"

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")


###Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark
####La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

movie_genre_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("genreId", IntegerType(), True)
])


In [0]:
#movie_genre_df = spark.read.schema(movie_genre_schema).json("abfss://bronze@moviee.dfs.core.windows.net/movie_genre.json")

movie_genre_df = spark.read.schema(movie_genre_schema).json(f"{bronze_folder_path}/movie_genre.json")

In [0]:
movie_genre_df.printSchema()

In [0]:
display(movie_genre_df)

### Paso 2 - Renombrar las columnas y añadir columnas nuevas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

#movie_genre_final_df = movie_genre_df \
#                        .withColumnRenamed("movieId", "movie_id") \
#                        .withColumnRenamed("genreId", "genre_id") \
#                        .withColumn("ingestion_date", current_timestamp()) \
#                        .withColumn("environment", lit("production"))


movie_genre_final_df = movie_genre_df \
                        .withColumnRenamed("movieId", "movie_id") \
                        .withColumnRenamed("genreId", "genre_id")

movie_genre_final_df = add_ingestion_date(movie_genre_final_df) \
                         .withColumn("environment", lit(v_environment))


In [0]:
display(movie_genre_final_df)

### Paso 3 - Escribir la salida en un formato "Parquet"

In [0]:
#movie_genre_final_df.write.mode("overwrite").partitionBy("movie_id").parquet("abfss://silver@moviee.dfs.core.windows.net/movie_genres")

movie_genre_final_df.write.mode("overwrite").partitionBy("movie_id").parquet(f"{silver_folder_path}/movie_genres")

In [0]:
#display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/movie_genres"))

display(spark.read.parquet(f"{silver_folder_path}/movie_genres"))

### Paso 4 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 3, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 3, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 3

In [0]:
movie_genre_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_genres")

In [0]:
%sql
SELECT * FROM movie_silver.movies_genres

In [0]:
dbutils.notebook.exit("Success")